# Phishing Website Detection — Step 3: Improving on the Baseline

**Capstone roadmap:**
1. ~~Dataset acquisition & exploration~~ ✅
2. ~~Baseline model + evaluation~~ ✅
3. **Step 3 (this notebook): try a stronger model, tune it, pick a final model**
4. Wrap model in an API
5. Build the browser extension

### Your Step 2 baseline (Logistic Regression) — the number to beat

| Metric | Value |
|---|---|
| Accuracy | 0.93 |
| Precision (phishing) | 0.94 |
| Recall (phishing) | 0.90 |
| F1 (phishing) | 0.92 |

We especially want to push **recall on the phishing class** up — that's the metric tied to phishing sites slipping through undetected.

### What this notebook covers
- Training a **Random Forest** on the exact same split, so the comparison is fair
- Reading **feature importance** and comparing it to Step 2's coefficients
- A light, understandable pass at **hyperparameter tuning**
- Saving the final chosen model to disk for the API in Step 4

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")

## 1. Load data and recreate the *exact same* split

This is important: to fairly compare Random Forest against your Step 2 Logistic Regression numbers, both models need to be trained and tested on identical data. Using the same `test_size`, `stratify`, and `random_state=42` as Step 2 guarantees this — it's the same split, byte for byte.

In [ ]:
df = pd.read_csv('phishing_data.csv')
X = df.drop(columns=['Result'])
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training samples:", X_train.shape[0], "| Test samples:", X_test.shape[0])

## 2. What is a Random Forest, briefly?

A single **decision tree** asks a sequence of yes/no questions about the features ("Is the URL length > X? Is there an `@` symbol?") to arrive at a prediction. One tree alone tends to overfit — it memorizes quirks of the training data.

A **Random Forest** trains *many* decision trees, each on a random subset of the data and features, then lets them **vote** on the final prediction. This averaging reduces overfitting and usually captures more complex, non-linear patterns than a single linear model like Logistic Regression can.

Key parameters we'll use:
- `n_estimators`: how many trees to grow
- `max_depth`: how deep each tree can grow (deeper = more complex = more prone to overfitting)
- `random_state=42`: reproducibility, same as before

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

## 3. Evaluate — and compare directly against Step 2

In [ ]:
print("=== Random Forest (default settings) ===")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_rf), 3))
print("Precision (phishing):", round(precision_score(y_test, y_pred_rf, pos_label=-1), 3))
print("Recall (phishing):   ", round(recall_score(y_test, y_pred_rf, pos_label=-1), 3))
print("F1 (phishing):        ", round(f1_score(y_test, y_pred_rf, pos_label=-1), 3))

print("\n=== Your Step 2 Logistic Regression baseline (for reference) ===")
print("Accuracy: 0.93 | Precision: 0.94 | Recall: 0.90 | F1: 0.92")

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=[-1, 1])

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_rf, annot=True, fmt='d', cmap='Greens',
    xticklabels=['Predicted: Phishing', 'Predicted: Legitimate'],
    yticklabels=['Actual: Phishing', 'Actual: Legitimate']
)
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

**Compare this to your Step 2 confusion matrix**: did the false negative count (real phishing sites predicted as legitimate) go down? That's the improvement that matters most for this problem.

## 4. Feature importance — does Random Forest agree with Logistic Regression?

Random Forest computes **feature importance** differently than Logistic Regression's coefficients (it's based on how much each feature reduces impurity across all the trees), but conceptually it answers a similar question: *which features actually drive the prediction?*

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(10).plot(kind='barh')
plt.title('Top 10 most important features (Random Forest)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()

importances.head(10)

**Your task:** open your Step 2 notebook and compare its top-10 list to this one. Note here (edit this cell) which features show up on *both* lists — those are probably genuinely strong phishing signals, agreed on by two very different types of model.

## 5. Light hyperparameter tuning

So far we used default settings (`n_estimators=200`, no depth limit). **Hyperparameters** are settings we choose *before* training — unlike the model's learned weights, they're not learned from data automatically. Picking good ones can meaningfully improve performance.

`GridSearchCV` tries every combination of the parameter values we give it, using **cross-validation** (splitting the training data into folds and rotating which fold is held out, to get a more reliable performance estimate than a single split) to pick the best combination — all without ever touching our held-out test set.

This search can take a minute or two to run — that's expected, it's training many forests.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,                # 5-fold cross-validation
    scoring='f1_macro',  # optimize for balanced performance across both classes
    n_jobs=-1             # use all available CPU cores
)

grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
print("Best cross-validation F1 score:", round(grid_search.best_score_, 3))

In [ ]:
# Use the best model found by the search to predict on the test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("=== Tuned Random Forest ===")
print("Accuracy: ", round(accuracy_score(y_test, y_pred_best), 3))
print("Precision (phishing):", round(precision_score(y_test, y_pred_best, pos_label=-1), 3))
print("Recall (phishing):   ", round(recall_score(y_test, y_pred_best, pos_label=-1), 3))
print("F1 (phishing):        ", round(f1_score(y_test, y_pred_best, pos_label=-1), 3))

**Note:** tuning doesn't always produce a dramatic jump — sometimes the default settings were already close to optimal for this dataset. That's a normal, useful finding, not a failure. What matters is that you now understand *how* to search for better settings, and how to verify whether it actually helped.

## 6. Pick your final model and save it

Compare all three sets of numbers (Logistic Regression baseline, default Random Forest, tuned Random Forest) and decide which one you're taking forward. In most cases here, the tuned Random Forest should win — but check your actual numbers rather than assuming.

We save the model using `joblib`, which serializes the trained model object to a `.pkl` file. This is what Step 4's API will load — so it doesn't need to retrain the model every time it starts up.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)  # adjust path if your notebook lives elsewhere

joblib.dump(best_rf, '../models/phishing_model.pkl')

# Also save the exact list and order of feature columns —
# the API will need to build input in this exact same order
joblib.dump(list(X.columns), '../models/feature_columns.pkl')

print("Saved model to models/phishing_model.pkl")
print("Saved feature column order to models/feature_columns.pkl")

## Summary — what we now have

- [ ] A Random Forest trained and evaluated on the exact same split as your baseline
- [ ] A direct, fair comparison between Logistic Regression and Random Forest
- [ ] An understanding of hyperparameter tuning via cross-validation, without touching the test set
- [ ] A saved, ready-to-load model file in `models/phishing_model.pkl`

**Your task before Step 4:** fill in your final numbers here so we have them on record:

- Final model chosen: ___
- Accuracy: ___
- Precision (phishing): ___
- Recall (phishing): ___
- F1 (phishing): ___

**Next up (Step 4):** we'll build a small API (using Flask) that loads `phishing_model.pkl` and takes requests. There's an important catch we'll need to solve: this dataset's 30 features were *pre-extracted* by someone else. Our API will need to extract a usable feature set live from a raw URL a user types in — that's a real, interesting problem we'll tackle together.